In [4]:
import gdsfactory as gf

from mesalab_pdk import get_pdk, LAYER

pdk = get_pdk()
pdk.activate()

print("PDK name:", pdk.name)
print("Has xs_heater_metal_trench:", "xs_heater_metal_trench" in pdk.cross_sections)
print("Has xs_ekn300_te_IMGREV:", "xs_ekn300_te_IMGREV" in pdk.cross_sections)
print("HEATER layer:", LAYER.HEATER)

c = gf.Component("pdk_smoke_test")

wg = c << gf.components.straight(length=50, cross_section="xs_ekn300_te_IMGREV")
wg.move((0, 0))

heater = c << gf.components.straight(length=30, cross_section="xs_heater_metal_trench")
heater.move((0, 20))

c.pprint_ports()
c.show()

PDK name: mesapdk_lab
Has xs_heater_metal_trench: True
Has xs_ekn300_te_IMGREV: True


AttributeError: type object 'LAYER' has no attribute 'HEATER'

In [ ]:

import gdsfactory as gf

from mesalab_pdk import get_pdk, LAYER
pdk = get_pdk()
pdk.activate()

c = gf.Component()

c.add_ref(gf.components.straight(length=40, cross_section="xs_ekn300_te_IMGREV"))
c.add_ref(gf.components.straight(length=20, cross_section="xs_heater_metal_trench")).movey(25)

layers = sorted(c.layers)
print("Layers in component:", layers)

wg_layer = gf.get_layer(LAYER.WG)
heater_layer = gf.get_layer(LAYER.HEATER)

print("Resolved WG layer:", wg_layer)
print("Resolved HEATER layer:", heater_layer)

assert wg_layer in layers, f"Missing WG layer {wg_layer}"
assert heater_layer in layers, f"Missing HEATER layer {heater_layer}"

c.show()

Layers in component: [(1, 0), (3, 6), (4, 0), (47, 0)]
Resolved WG layer: WG
Resolved HEATER layer: HEATER


AssertionError: Missing WG layer WG

In [ ]:
import gdsfactory as gf

from mesalab_pdk import get_pdk, LAYER
pdk = get_pdk()
pdk.activate()

LAYER_VIEWS = pdk.get_layer_views()
c = LAYER_VIEWS.preview_layerset()
c.show()

In [ ]:
from gdsfactory.typings import LayerSpec
@gf.cell
def dc_straight_slot_ekn300_IMGREV(
    length: float = 50,
    width: float = 0.75,
    gap: float = 0.1,  # um
    width_trench: float = 15,
    layer: LayerSpec = "WG",
    layer_trench: LayerSpec = "SIN_ETCH",
) -> gf.Component:
    c = gf.Component()

    pitch = width + gap
    xs_core = gf.cross_section.strip(width=width, layer=layer)

    top = c << gf.components.straight(length=length, cross_section=xs_core)
    bot = c << gf.components.straight(length=length, cross_section=xs_core)

    top.movey(+pitch / 2)
    bot.movey(-pitch / 2)

    # central etched slot
    c.add_polygon(
        [
            (0, -gap / 2),
            (length, -gap / 2),
            (length, +gap / 2),
            (0, +gap / 2),
        ],
        layer=layer_trench,
    )

    # outer trenches
    y_top_trench = +pitch / 2 + width / 2 + width_trench / 2
    y_bot_trench = -pitch / 2 - width / 2 - width_trench / 2

    for y in [y_top_trench, y_bot_trench]:
        c.add_polygon(
            [
                (0, y - width_trench / 2),
                (length, y - width_trench / 2),
                (length, y + width_trench / 2),
                (0, y + width_trench / 2),
            ],
            layer=layer_trench,
        )

    c.add_port("o1", port=bot.ports["o1"])
    c.add_port("o2", port=top.ports["o1"])
    c.add_port("o3", port=top.ports["o2"])
    c.add_port("o4", port=bot.ports["o2"])

    return c

In [ ]:
c = dc_straight_slot_ekn300_IMGREV()
c.show()

In [7]:
from __future__ import annotations

import numpy as np
import gdsfactory as gf
from gdsfactory.typings import LayerSpec


def _smoothstep(t: np.ndarray) -> np.ndarray:
    return t * t * (3 - 2 * t)


def _add_ribbon(
    c: gf.Component,
    x: np.ndarray,
    y_lower: np.ndarray,
    y_upper: np.ndarray,
    layer: LayerSpec,
) -> None:
    """Adds polygon ribbon between lower and upper y-boundaries."""
    pts_lower = list(zip(x, y_lower))
    pts_upper = list(zip(x[::-1], y_upper[::-1]))
    c.add_polygon(pts_lower + pts_upper, layer=layer)


def _segments(mask: np.ndarray) -> list[tuple[int, int]]:
    """Returns contiguous True segments as [i0, i1] inclusive."""
    out = []
    i = 0
    n = len(mask)

    while i < n:
        if not mask[i]:
            i += 1
            continue

        j = i
        while j + 1 < n and mask[j + 1]:
            j += 1

        if j > i:
            out.append((i, j))

        i = j + 1

    return out


@gf.cell
def dc_trench_adaptive_IMGREV(
    length_sbend: float = 80.0,
    length_coupler: float = 50.0,
    width: float = 0.75,
    gap: float = 0.10,
    pitch_in: float = 35.0,
    width_trench: float = 15.0,
    layer: LayerSpec = "WG",
    layer_trench: LayerSpec = "SIN_ETCH",
    npoints_sbend: int = 151,
    npoints_coupler: int = 51,
    add_inner_trenches_before_merge: bool = True,
) -> gf.Component:
    """Directional coupler for image-reversal trench-defined SiN waveguides.

    Geometry model:
        - two SiN cores follow smooth S-bend approach paths
        - outer trenches remain independent and constant-width
        - inner trenches are independent while they fit
        - once inner trenches collide, they become one shared central etched slot

    Important:
        gap is in um, so 50 nm = 0.05, 100 nm = 0.10, 250 nm = 0.25.
    """
    if gap <= 0:
        raise ValueError("gap must be positive.")

    if gap < 0.05:
        raise ValueError(
            f"gap={gap} um is below 50 nm. Do not generate fantasy geometry."
        )

    pitch_dc = width + gap

    if pitch_in <= pitch_dc:
        raise ValueError("pitch_in must be larger than width + gap.")

    if pitch_in < width + 2 * width_trench + gap:
        raise ValueError(
            "pitch_in is too small for two isolated trench waveguides. "
            f"Need roughly >= {width + 2 * width_trench + gap:.3f} um."
        )

    c = gf.Component()

    # ------------------------------------------------------------------
    # x-grid
    # ------------------------------------------------------------------
    x_left = np.linspace(0, length_sbend, npoints_sbend)
    x_mid = np.linspace(
        length_sbend,
        length_sbend + length_coupler,
        npoints_coupler,
    )[1:]
    x_right = np.linspace(
        length_sbend + length_coupler,
        2 * length_sbend + length_coupler,
        npoints_sbend,
    )[1:]

    x = np.concatenate([x_left, x_mid, x_right])

    # ------------------------------------------------------------------
    # pitch profile
    # ------------------------------------------------------------------
    t_left = x_left / length_sbend
    pitch_left = pitch_in + (pitch_dc - pitch_in) * _smoothstep(t_left)

    pitch_mid = np.full(len(x_mid), pitch_dc)

    t_right = (x_right - (length_sbend + length_coupler)) / length_sbend
    pitch_right = pitch_dc + (pitch_in - pitch_dc) * _smoothstep(t_right)

    pitch = np.concatenate([pitch_left, pitch_mid, pitch_right])

    y_top = +pitch / 2
    y_bot = -pitch / 2

    # ------------------------------------------------------------------
    # Core polygons
    # ------------------------------------------------------------------
    _add_ribbon(
        c,
        x,
        y_top - width / 2,
        y_top + width / 2,
        layer=layer,
    )

    _add_ribbon(
        c,
        x,
        y_bot - width / 2,
        y_bot + width / 2,
        layer=layer,
    )

    # ------------------------------------------------------------------
    # Outer trenches
    # ------------------------------------------------------------------
    _add_ribbon(
        c,
        x,
        y_top + width / 2,
        y_top + width / 2 + width_trench,
        layer=layer_trench,
    )

    _add_ribbon(
        c,
        x,
        y_bot - width / 2 - width_trench,
        y_bot - width / 2,
        layer=layer_trench,
    )

    # ------------------------------------------------------------------
    # Inner trench / shared slot logic
    # ------------------------------------------------------------------
    top_inner_edge = y_top - width / 2
    bot_inner_edge = y_bot + width / 2

    available_inner_space = top_inner_edge - bot_inner_edge

    separated = available_inner_space > 2 * width_trench
    merged = ~separated

    if add_inner_trenches_before_merge:
        for i0, i1 in _segments(separated):
            xs = x[i0 : i1 + 1]

            # Top arm inner trench: below top core
            _add_ribbon(
                c,
                xs,
                top_inner_edge[i0 : i1 + 1] - width_trench,
                top_inner_edge[i0 : i1 + 1],
                layer=layer_trench,
            )

            # Bottom arm inner trench: above bottom core
            _add_ribbon(
                c,
                xs,
                bot_inner_edge[i0 : i1 + 1],
                bot_inner_edge[i0 : i1 + 1] + width_trench,
                layer=layer_trench,
            )

    # Shared etched region between the two inner core edges.
    # In the coupler middle this becomes exactly `gap`.
    for i0, i1 in _segments(merged):
        _add_ribbon(
            c,
            x[i0 : i1 + 1],
            bot_inner_edge[i0 : i1 + 1],
            top_inner_edge[i0 : i1 + 1],
            layer=layer_trench,
        )

    # ------------------------------------------------------------------
    # Optical ports
    # ------------------------------------------------------------------
    x0 = 0
    x1 = 2 * length_sbend + length_coupler

    c.add_port(
        "o1",
        center=(x0, -pitch_in / 2),
        width=width,
        orientation=180,
        layer=layer,
        port_type="optical",
    )
    c.add_port(
        "o2",
        center=(x0, +pitch_in / 2),
        width=width,
        orientation=180,
        layer=layer,
        port_type="optical",
    )
    c.add_port(
        "o3",
        center=(x1, +pitch_in / 2),
        width=width,
        orientation=0,
        layer=layer,
        port_type="optical",
    )
    c.add_port(
        "o4",
        center=(x1, -pitch_in / 2),
        width=width,
        orientation=0,
        layer=layer,
        port_type="optical",
    )

    return c

In [8]:
c = dc_trench_adaptive_IMGREV(
    gap=0.05,          # 50 nm
    length_sbend=120,
    length_coupler=40,
    pitch_in=35,
)
c.show()

In [9]:
from __future__ import annotations

from collections.abc import Callable
import numpy as np
import gdsfactory as gf
from gdsfactory.typings import CrossSectionSpec, LayerSpec


def smoothstep(t: np.ndarray) -> np.ndarray:
    return t * t * (3 - 2 * t)


def sine_smooth(t: np.ndarray) -> np.ndarray:
    return 0.5 - 0.5 * np.cos(np.pi * t)


def _get_section_by_name(xs: gf.CrossSection, name: str):
    for section in xs.sections:
        if section.name == name:
            return section
    return None


def _add_ribbon(
    c: gf.Component,
    x: np.ndarray,
    y_lower: np.ndarray,
    y_upper: np.ndarray,
    layer: LayerSpec,
) -> None:
    """Adds polygon ribbon between lower and upper y boundaries."""
    if len(x) < 2:
        return

    pts_lower = list(zip(x, y_lower))
    pts_upper = list(zip(x[::-1], y_upper[::-1]))
    c.add_polygon(pts_lower + pts_upper, layer=layer)


def _resolve_bend_function(
    bend_function: str | Callable[[np.ndarray], np.ndarray],
) -> Callable[[np.ndarray], np.ndarray]:
    if callable(bend_function):
        return bend_function

    if bend_function == "smoothstep":
        return smoothstep

    if bend_function == "sine":
        return sine_smooth

    raise ValueError(
        f"Unsupported bend_function={bend_function!r}. "
        "Use 'smoothstep', 'sine', or a callable."
    )


@gf.cell
def dc_trench_adaptive_IMGREV_v2(
    length_sbend: float = 80.0,
    length_coupler: float = 50.0,
    gap: float = 0.10,
    pitch_in: float = 35.0,
    cross_section: CrossSectionSpec = "xs_ekn300_te_IMGREV",
    bend_function: str | Callable[[np.ndarray], np.ndarray] = "sine",
    npoints_sbend: int = 201,
    npoints_coupler: int = 101,
    layer_trench: LayerSpec | None = None,
    width_trench: float | None = None,
) -> gf.Component:
    """Directional coupler for trench-defined SiN waveguides.

    The optical arms follow a smooth S-bend pitch profile.

    Geometry:
        - WG cores are drawn from the resolved cross_section width/layer.
        - Outer trenches are kept constant.
        - The inner region is drawn as one continuous shared etched slot.
          This avoids the unetched seam at the merge point.

    Args:
        length_sbend: Length of one S-bend transition.
        length_coupler: Length of straight coupling region.
        gap: Final etched gap between the two SiN cores, in um.
        pitch_in: Input/output center-to-center pitch.
        cross_section: Normal isolated waveguide cross-section.
        bend_function: 'sine', 'smoothstep', or callable f(t), t in [0, 1].
        npoints_sbend: Sampling points per S-bend.
        npoints_coupler: Sampling points in straight coupler.
        layer_trench: Optional override for trench layer.
        width_trench: Optional override for trench width.
    """
    if gap <= 0:
        raise ValueError("gap must be positive.")

    if gap < 0.05:
        raise ValueError(
            f"gap={gap} um is below 50 nm. That is not a sane drawn split."
        )

    if npoints_sbend < 5:
        raise ValueError("npoints_sbend must be >= 5.")

    if npoints_coupler < 2:
        raise ValueError("npoints_coupler must be >= 2.")

    xs = gf.get_cross_section(cross_section)

    width = float(xs.width)
    layer = xs.layer

    # Try to infer trench settings from your existing xs_ekn300_te_IMGREV.
    trench_top = _get_section_by_name(xs, "trench_top")
    trench_bot = _get_section_by_name(xs, "trench_bot")

    trench_section = trench_top or trench_bot

    if trench_section is None and (layer_trench is None or width_trench is None):
        raise ValueError(
            "Could not infer trench section from cross_section. "
            "Expected section named 'trench_top' or 'trench_bot', "
            "or pass layer_trench and width_trench explicitly."
        )

    if layer_trench is None:
        layer_trench = trench_section.layer

    if width_trench is None:
        width_trench = float(trench_section.width)

    pitch_dc = width + gap

    if pitch_in <= pitch_dc:
        raise ValueError(
            f"pitch_in={pitch_in} must be larger than final pitch {pitch_dc}."
        )

    c = gf.Component()

    bend = _resolve_bend_function(bend_function)

    # ------------------------------------------------------------
    # Build x-grid
    # ------------------------------------------------------------
    x_left = np.linspace(0, length_sbend, npoints_sbend)

    x_mid = np.linspace(
        length_sbend,
        length_sbend + length_coupler,
        npoints_coupler,
    )[1:]

    x_right = np.linspace(
        length_sbend + length_coupler,
        2 * length_sbend + length_coupler,
        npoints_sbend,
    )[1:]

    x = np.concatenate([x_left, x_mid, x_right])

    # ------------------------------------------------------------
    # Pitch profile: pitch_in -> width + gap -> pitch_in
    # ------------------------------------------------------------
    t_left = x_left / length_sbend
    f_left = np.asarray(bend(t_left), dtype=float)

    t_right = (x_right - (length_sbend + length_coupler)) / length_sbend
    f_right = np.asarray(bend(t_right), dtype=float)

    if f_left.shape != t_left.shape:
        raise ValueError("bend_function must return array with same shape as input.")

    if f_right.shape != t_right.shape:
        raise ValueError("bend_function must return array with same shape as input.")

    pitch_left = pitch_in + (pitch_dc - pitch_in) * f_left
    pitch_mid = np.full(len(x_mid), pitch_dc)
    pitch_right = pitch_dc + (pitch_in - pitch_dc) * f_right

    pitch = np.concatenate([pitch_left, pitch_mid, pitch_right])

    y_top = +pitch / 2
    y_bot = -pitch / 2

    # ------------------------------------------------------------
    # Core polygons
    # ------------------------------------------------------------
    top_core_lower = y_top - width / 2
    top_core_upper = y_top + width / 2

    bot_core_lower = y_bot - width / 2
    bot_core_upper = y_bot + width / 2

    _add_ribbon(c, x, top_core_lower, top_core_upper, layer=layer)
    _add_ribbon(c, x, bot_core_lower, bot_core_upper, layer=layer)

    # ------------------------------------------------------------
    # Outer trenches
    # ------------------------------------------------------------
    _add_ribbon(
        c,
        x,
        top_core_upper,
        top_core_upper + width_trench,
        layer=layer_trench,
    )

    _add_ribbon(
        c,
        x,
        bot_core_lower - width_trench,
        bot_core_lower,
        layer=layer_trench,
    )

    # ------------------------------------------------------------
    # Continuous shared inner etched region
    #
    # This is the actual bug fix:
    # no split into "separated" / "merged" segments,
    # therefore no vertical unetched seam.
    # ------------------------------------------------------------
    _add_ribbon(
        c,
        x,
        bot_core_upper,
        top_core_lower,
        layer=layer_trench,
    )

    # ------------------------------------------------------------
    # Optical ports
    # ------------------------------------------------------------
    x0 = 0.0
    x1 = 2 * length_sbend + length_coupler

    c.add_port(
        "o1",
        center=(x0, -pitch_in / 2),
        width=width,
        orientation=180,
        layer=layer,
        port_type="optical",
        cross_section=xs,
    )

    c.add_port(
        "o2",
        center=(x0, +pitch_in / 2),
        width=width,
        orientation=180,
        layer=layer,
        port_type="optical",
        cross_section=xs,
    )

    c.add_port(
        "o3",
        center=(x1, +pitch_in / 2),
        width=width,
        orientation=0,
        layer=layer,
        port_type="optical",
        cross_section=xs,
    )

    c.add_port(
        "o4",
        center=(x1, -pitch_in / 2),
        width=width,
        orientation=0,
        layer=layer,
        port_type="optical",
        cross_section=xs,
    )

    return c

In [11]:
from cross_sections import xs_ekn300_te_IMGREV

xs = xs_ekn300_te_IMGREV(
    width=0.75,
    width_trench=15,
    layer="WG",
    layer_trench="SIN_ETCH",
)

c = dc_trench_adaptive_IMGREV_v2(
    cross_section=xs,
    gap=0.10,
    pitch_in=80,
    length_sbend=250,
    length_coupler=50,
    bend_function="sine",
)

c.show()

In [12]:
from __future__ import annotations

from collections.abc import Callable
import numpy as np
import gdsfactory as gf
from gdsfactory.typings import CrossSectionSpec, LayerSpec


def _smoothstep(t: np.ndarray) -> np.ndarray:
    return t * t * (3 - 2 * t)


def _sine(t: np.ndarray) -> np.ndarray:
    return 0.5 - 0.5 * np.cos(np.pi * t)


def _resolve_profile(
    profile: str | Callable[[np.ndarray], np.ndarray],
) -> Callable[[np.ndarray], np.ndarray]:
    if callable(profile):
        return profile

    profiles = {
        "sine": _sine,
        "smoothstep": _smoothstep,
    }

    if profile not in profiles:
        raise ValueError(f"Unknown profile {profile!r}. Use 'sine', 'smoothstep', or callable.")

    return profiles[profile]


def _get_named_section(xs: gf.CrossSection, name: str):
    for section in xs.sections:
        if section.name == name:
            return section
    return None


def _add_ribbon(
    c: gf.Component,
    x: np.ndarray,
    y_lower: np.ndarray,
    y_upper: np.ndarray,
    layer: LayerSpec,
) -> None:
    """Adds one polygon ribbon between y_lower(x) and y_upper(x)."""
    if len(x) < 2:
        return

    if np.any(y_upper < y_lower):
        raise ValueError("Invalid ribbon: y_upper is below y_lower.")

    pts = list(zip(x, y_lower)) + list(zip(x[::-1], y_upper[::-1]))
    c.add_polygon(pts, layer=layer)


def _extract_trench_params(
    xs: gf.CrossSection,
    layer_trench: LayerSpec | None = None,
    width_trench: float | None = None,
) -> tuple[LayerSpec, float]:
    """Extract trench layer and width from xs sections."""
    trench_top = _get_named_section(xs, "trench_top")
    trench_bot = _get_named_section(xs, "trench_bot")
    trench = trench_top or trench_bot

    if trench is None:
        if layer_trench is None or width_trench is None:
            raise ValueError(
                "Could not infer trench parameters. "
                "Expected cross_section sections named 'trench_top'/'trench_bot', "
                "or pass layer_trench and width_trench explicitly."
            )
        return layer_trench, float(width_trench)

    if layer_trench is None:
        layer_trench = trench.layer

    if width_trench is None:
        width_trench = float(trench.width)

    return layer_trench, float(width_trench)


@gf.cell
def dc_trench_adaptive_IMGREV(
    length_sbend: float = 80.0,
    length_coupler: float = 50.0,
    gap: float = 0.10,
    pitch_in: float = 35.0,
    cross_section: CrossSectionSpec = "xs_ekn300_te_IMGREV",
    profile: str | Callable[[np.ndarray], np.ndarray] = "sine",
    npoints_sbend: int = 201,
    npoints_coupler: int = 101,
    layer_trench: LayerSpec | None = None,
    width_trench: float | None = None,
) -> gf.Component:
    """Directional coupler for trench-defined image-reversal waveguides.

    Geometry:
        far from coupler:
            outer trench | core | inner trench | unetched middle | inner trench | core | outer trench

        near/inside coupler:
            outer trench | core | capped shared etched gap | core | outer trench

    Important:
        gap is in um:
            50 nm  -> 0.05
            100 nm -> 0.10
            250 nm -> 0.25

    Args:
        length_sbend: Length of each transition section.
        length_coupler: Straight coupling length.
        gap: Final etched gap between waveguide cores.
        pitch_in: Input/output center-to-center pitch.
        cross_section: Isolated arm cross-section. Used for core width/layer and trench info.
        profile: 'sine', 'smoothstep', or callable f(t) mapping [0,1] -> [0,1].
        npoints_sbend: Sampling points for each transition.
        npoints_coupler: Sampling points for straight coupling region.
        layer_trench: Optional trench layer override.
        width_trench: Optional trench width override.
    """
    xs = gf.get_cross_section(cross_section)

    width = float(xs.width)
    layer = xs.layer
    port_type = xs.port_types[0] if getattr(xs, "port_types", None) else "optical"

    layer_trench, width_trench = _extract_trench_params(
        xs=xs,
        layer_trench=layer_trench,
        width_trench=width_trench,
    )

    if gap <= 0:
        raise ValueError("gap must be positive.")

    if gap < 0.05:
        raise ValueError(f"gap={gap} um is below 50 nm.")

    pitch_dc = width + gap

    if pitch_in <= pitch_dc:
        raise ValueError(
            f"pitch_in={pitch_in} must be larger than final pitch {pitch_dc}."
        )

    if npoints_sbend < 5:
        raise ValueError("npoints_sbend must be >= 5.")

    if npoints_coupler < 2:
        raise ValueError("npoints_coupler must be >= 2.")

    f = _resolve_profile(profile)

    c = gf.Component()

    # x coordinates
    x_left = np.linspace(0, length_sbend, npoints_sbend)
    x_mid = np.linspace(
        length_sbend,
        length_sbend + length_coupler,
        npoints_coupler,
    )[1:]
    x_right = np.linspace(
        length_sbend + length_coupler,
        2 * length_sbend + length_coupler,
        npoints_sbend,
    )[1:]

    x = np.concatenate([x_left, x_mid, x_right])

    # pitch profile
    t_left = x_left / length_sbend
    t_right = (x_right - length_sbend - length_coupler) / length_sbend

    f_left = np.asarray(f(t_left), dtype=float)
    f_right = np.asarray(f(t_right), dtype=float)

    if f_left.shape != t_left.shape or f_right.shape != t_right.shape:
        raise ValueError("profile function must return an array with the same shape as input.")

    if np.any(f_left < -1e-9) or np.any(f_left > 1 + 1e-9):
        raise ValueError("profile function must stay in [0, 1].")

    if np.any(f_right < -1e-9) or np.any(f_right > 1 + 1e-9):
        raise ValueError("profile function must stay in [0, 1].")

    pitch_left = pitch_in + (pitch_dc - pitch_in) * f_left
    pitch_mid = np.full(len(x_mid), pitch_dc)
    pitch_right = pitch_dc + (pitch_in - pitch_dc) * f_right

    pitch = np.concatenate([pitch_left, pitch_mid, pitch_right])

    y_top = +pitch / 2
    y_bot = -pitch / 2

    # core boundaries
    top_core_lower = y_top - width / 2
    top_core_upper = y_top + width / 2
    bot_core_lower = y_bot - width / 2
    bot_core_upper = y_bot + width / 2

    # WG cores
    _add_ribbon(c, x, top_core_lower, top_core_upper, layer=layer)
    _add_ribbon(c, x, bot_core_lower, bot_core_upper, layer=layer)

    # outer trenches
    _add_ribbon(
        c,
        x,
        top_core_upper,
        top_core_upper + width_trench,
        layer=layer_trench,
    )

    _add_ribbon(
        c,
        x,
        bot_core_lower - width_trench,
        bot_core_lower,
        layer=layer_trench,
    )

    # adaptive inner trenches
    available_inner_space = top_core_lower - bot_core_upper

    if np.any(available_inner_space <= 0):
        raise ValueError("Waveguide cores overlap. Increase pitch or reduce width/gap.")

    inner_etch_width = np.minimum(width_trench, available_inner_space / 2)

    # bottom inner trench
    _add_ribbon(
        c,
        x,
        bot_core_upper,
        bot_core_upper + inner_etch_width,
        layer=layer_trench,
    )

    # top inner trench
    _add_ribbon(
        c,
        x,
        top_core_lower - inner_etch_width,
        top_core_lower,
        layer=layer_trench,
    )

    # ports
    x0 = 0.0
    x1 = 2 * length_sbend + length_coupler

    c.add_port(
        "o1",
        center=(x0, -pitch_in / 2),
        width=width,
        orientation=180,
        layer=layer,
        port_type=port_type,
        cross_section=xs,
    )

    c.add_port(
        "o2",
        center=(x0, +pitch_in / 2),
        width=width,
        orientation=180,
        layer=layer,
        port_type=port_type,
        cross_section=xs,
    )

    c.add_port(
        "o3",
        center=(x1, +pitch_in / 2),
        width=width,
        orientation=0,
        layer=layer,
        port_type=port_type,
        cross_section=xs,
    )

    c.add_port(
        "o4",
        center=(x1, -pitch_in / 2),
        width=width,
        orientation=0,
        layer=layer,
        port_type=port_type,
        cross_section=xs,
    )

    return c

In [14]:
xs = xs_ekn300_te_IMGREV(
    width=0.75,
    width_trench=15,
    layer="WG",
    layer_trench="SIN_ETCH",
)

c = dc_trench_adaptive_IMGREV(
    cross_section=xs,
    gap=0.10,
    pitch_in=75,
    length_sbend=250,
    length_coupler=50,
    profile="sine",
)

c.show()

In [ ]:
gf.components.coupler()

In [15]:
from __future__ import annotations

import gdsfactory as gf
from gdsfactory.typings import CrossSectionSpec, LayerSpec


def _get_layer_from_xs(xs: gf.CrossSection) -> LayerSpec:
    return xs.layer


def _get_trench_layer_from_xs(
    xs: gf.CrossSection,
    section_names: tuple[str, ...] = ("trench_top", "trench_bot"),
) -> LayerSpec:
    for section in xs.sections:
        if section.name in section_names:
            return section.layer

    raise ValueError(
        f"Could not infer trench layer. Expected one of {section_names} "
        "in cross_section.sections."
    )


def _copy_ports(dst: gf.Component, src: gf.Component) -> None:
    for name, port in src.ports.items():
        dst.add_port(name=name, port=port)


@gf.cell
def _clean_trenches_after_boolean(
    component: gf.Component,
    layer_core: LayerSpec,
    layer_trench: LayerSpec,
) -> gf.Component:
    """Merge trench polygons and remove any trench area overlapping the core."""
    c = gf.Component()

    core = component.extract(layers=[layer_core])
    trench = component.extract(layers=[layer_trench])

    trench_merged = gf.boolean(
        A=trench,
        B=None,
        operation="or",
        layer=layer_trench,
    )

    trench_clean = gf.boolean(
        A=trench_merged,
        B=core,
        operation="not",
        layer=layer_trench,
    )

    c << core
    c << trench_clean

    _copy_ports(c, component)
    return c


@gf.cell
def coupler_trenched_postprocessed(
    gap: float = 0.10,
    length: float = 20.0,
    dy: float = 4.0,
    dx: float = 10.0,
    cross_section: CrossSectionSpec = "xs_ekn300_te_IMGREV",
    layer_core: LayerSpec | None = None,
    layer_trench: LayerSpec | None = None,
) -> gf.Component:
    """Directional coupler using normal trench waveguide XS + boolean cleanup.

    Similar spirit to gf.components.coupler, but intended for image-reversal /
    trench-defined waveguides.

    Args:
        gap: Final gap between SiN cores in the straight coupling section.
        length: Straight coupling length.
        dy: Input/output vertical separation between waveguide centers.
        dx: Horizontal length of each S-bend.
        cross_section: Normal isolated trench waveguide cross-section.
        layer_core: Optional core layer override.
        layer_trench: Optional trench layer override.

    Ports:
        o1, o2 on the left.
        o3, o4 on the right.
    """
    xs = gf.get_cross_section(cross_section)
    width = float(xs.width)

    if layer_core is None:
        layer_core = _get_layer_from_xs(xs)

    if layer_trench is None:
        layer_trench = _get_trench_layer_from_xs(xs)

    if gap <= 0:
        raise ValueError("gap must be positive.")

    if gap < 0.05:
        raise ValueError(f"gap={gap} um is below 50 nm.")

    if dy <= width + gap:
        raise ValueError(
            f"dy={dy} must be larger than final pitch {width + gap}."
        )

    raw = gf.Component()

    pitch_dc = width + gap
    y_left_top = +dy / 2
    y_left_bot = -dy / 2
    y_dc_top = +pitch_dc / 2
    y_dc_bot = -pitch_dc / 2

    # ------------------------------------------------------------------
    # Build paths
    # ------------------------------------------------------------------
    p_top = gf.Path()
    p_top += gf.path.straight(length=dx / 2)
    p_top += gf.path.smooth(
        points=[
            (0, 0),
            (dx / 2, 0),
            (dx, y_dc_top - y_left_top),
        ]
    )
    p_top += gf.path.straight(length=length)
    p_top += gf.path.smooth(
        points=[
            (0, 0),
            (dx / 2, 0),
            (dx, y_left_top - y_dc_top),
        ]
    )
    p_top += gf.path.straight(length=dx / 2)

    p_bot = gf.Path()
    p_bot += gf.path.straight(length=dx / 2)
    p_bot += gf.path.smooth(
        points=[
            (0, 0),
            (dx / 2, 0),
            (dx, y_dc_bot - y_left_bot),
        ]
    )
    p_bot += gf.path.straight(length=length)
    p_bot += gf.path.smooth(
        points=[
            (0, 0),
            (dx / 2, 0),
            (dx, y_left_bot - y_dc_bot),
        ]
    )
    p_bot += gf.path.straight(length=dx / 2)

    top = raw << gf.path.extrude(p_top, cross_section=xs)
    bot = raw << gf.path.extrude(p_bot, cross_section=xs)

    top.movey(y_left_top)
    bot.movey(y_left_bot)

    # ------------------------------------------------------------------
    # Ports
    # ------------------------------------------------------------------
    raw.add_port("o1", port=bot.ports["o1"])
    raw.add_port("o2", port=top.ports["o1"])
    raw.add_port("o3", port=top.ports["o2"])
    raw.add_port("o4", port=bot.ports["o2"])

    raw.auto_rename_ports()

    return _clean_trenches_after_boolean(
        component=raw,
        layer_core=layer_core,
        layer_trench=layer_trench,
    )

In [16]:
xs = xs_ekn300_te_IMGREV(
    width=0.75,
    width_trench=15,
    layer="WG",
    layer_trench="SIN_ETCH",
)

c = coupler_trenched_postprocessed(
    gap=0.10,
    length=50,
    dx=80,
    dy=35,
    cross_section=xs,
)

c.show()

AttributeError: 'NoneType' object has no attribute 'is_regular_array'

In [18]:
c = gf.components.coupler_ring(radius=350,
                               cross_section=xs_ekn300_te_IMGREV,
                               cross_section_bend=xs_ekn300_te_IMGREV
                               )

c.show()

In [20]:
from directional_couplers import coupler_ring_imgrev

xs = xs_ekn300_te_IMGREV(
    width=0.75,
    width_trench=15,
    layer="WG",
    layer_trench="SIN_ETCH",
)

c = coupler_ring_imgrev(
    gap=0.1,
    radius=350,
    length_x=25,
    cross_section=xs,
)

c.show()

/home/sadilek/Dev/mesaplus/mesapdk-lab/.venv/lib/python3.13/site-packages/gdsfactory/components/couplers/coupler90.py:44: UserWarning: {'radius': 350} ignored for cross_section 'xs_ekn300_te_IMGREV'
  x = gf.get_cross_section(cross_section, radius=radius)


AttributeError: 'DPorts' object has no attribute 'items'